# Sprint 1 — Integration Effectiveness Test

**Goal**: Compare 4 conditions for the same complex question:

| Condition | What it does | LLM Calls |
|---|---|---|
| **A. Raw** | Send the bare question to the LLM | 1 |
| **B. Single Template** | Use the best single cognitive template | 1 |
| **C. Integrated Template** | Merge multiple templates into one, then execute | 2 (integrate + execute) |
| **D. Chain vs Single (CAI)** | Compare chain output vs single template | Uses CAI engine |

Each output is scored with the **Output Evaluator** (5 dimensions: Instruction Following, Reasoning Depth, Actionability, Structure Compliance, Cognitive Scaffolding).

---

## Prerequisites

1. Have an **OpenAI API key** set: `export OPENAI_API_KEY=sk-...` or set it in the cell below
2. Run from the repo root so `mycontext` is importable
3. Install deps: `pip install mycontext-ai litellm`

In [ ]:
import json
import os
import sys
import time

# Ensure the SDK is importable from the repo root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

# Set your API key here if not in environment
# os.environ['OPENAI_API_KEY'] = 'sk-...'

PROVIDER = 'openai'  # Change to 'anthropic' or 'gemini' to test other providers

## Step 1 — Import SDK Components

In [ ]:
from mycontext.core import Context
from mycontext.foundation import Directive
from mycontext.intelligence.chain_orchestration_agent import (
    PATTERN_BUILD_CONTEXT_REGISTRY,
)
from mycontext.intelligence.context_amplification import ContextAmplificationIndex
from mycontext.intelligence.output_evaluator import OutputEvaluator
from mycontext.intelligence.pattern_suggester import get_pattern_class
from mycontext.intelligence.quality_metrics import QualityMetrics
from mycontext.intelligence.template_integrator_agent import TemplateIntegratorAgent

evaluator = OutputEvaluator(mode='heuristic')
quality = QualityMetrics(mode='heuristic')
integrator = TemplateIntegratorAgent(include_enterprise=True)

print('All components loaded successfully.')

## Step 2 — Define Test Questions

10 diverse, complex questions spanning business, technical, ethical, and strategic domains.

In [ ]:
TEST_QUESTIONS = [
    "Why did customer churn spike 40% last quarter and what should we do about it?",
    "Should we migrate our monolithic architecture to microservices? What are the trade-offs?",
    "Our AI model is producing biased hiring recommendations. How do we diagnose and fix this?",
    "Design a go-to-market strategy for a B2B SaaS product entering the healthcare vertical.",
    "Our API response times tripled after the last deployment. Find the root cause and propose fixes.",
    "How should a startup allocate its $2M seed funding across engineering, marketing, and operations?",
    "Evaluate the ethical implications of using facial recognition in public schools.",
    "Our cross-functional team has persistent communication breakdowns. Diagnose and solve.",
    "Compare three database options (PostgreSQL, MongoDB, DynamoDB) for a real-time analytics platform.",
    "Develop a 12-month roadmap for transitioning from waterfall to agile across a 200-person org.",
]

# Start with the first 3 questions to save cost. Increase as needed.
ACTIVE_QUESTIONS = TEST_QUESTIONS[:3]
print(f'Testing {len(ACTIVE_QUESTIONS)} questions')

## Step 3 — Helper Functions

In [ ]:
def execute_raw(question, provider=PROVIDER):
    """Condition A: Send bare question to LLM."""
    ctx = Context(directive=Directive(content=question))
    t0 = time.time()
    result = ctx.execute(provider=provider)
    elapsed = time.time() - t0
    return ctx, result.response, elapsed


def execute_single_template(question, template_name, provider=PROVIDER):
    """Condition B: Use a single template."""
    klass = get_pattern_class(template_name, include_enterprise=True)
    if not klass:
        raise ValueError(f'Template not found: {template_name}')
    reg = PATTERN_BUILD_CONTEXT_REGISTRY.get(template_name, ('problem', {}))
    primary, defaults = reg
    params = dict(defaults)
    params[primary] = question
    ctx = klass().build_context(**params)
    t0 = time.time()
    result = ctx.execute(provider=provider)
    elapsed = time.time() - t0
    return ctx, result.response, elapsed


def execute_integrated(question, provider=PROVIDER, max_patterns=4):
    """Condition C: Suggest templates, integrate into one, then execute."""
    t0 = time.time()
    integration = integrator.suggest_and_integrate(
        question=question,
        provider=provider,
        max_patterns=max_patterns,
    )
    ctx = integration.to_context()
    result = ctx.execute(provider=provider)
    elapsed = time.time() - t0
    return ctx, result.response, elapsed, integration


def score_output(ctx, output):
    """Score an output against its context."""
    return evaluator.evaluate(ctx, output)


def format_score(score):
    """Pretty-print a score."""
    dims = {d.value: round(v * 100, 1) for d, v in score.dimensions.items()}
    return f'Overall: {score.overall:.1%} | ' + ' | '.join(f'{k}: {v}%' for k, v in dims.items())


print('Helpers ready.')

---

## Step 4 — Run the Full Comparison

This cell runs all 3 conditions for each question. **Expect ~30-60 seconds per question** (3 LLM calls each).

Results are stored in `results` for analysis.

In [ ]:
results = []

for i, question in enumerate(ACTIVE_QUESTIONS):
    print(f'\n{"="*70}')
    print(f'Question {i+1}/{len(ACTIVE_QUESTIONS)}: {question[:80]}...')
    print('='*70)
    row = {'question': question}

    # --- Condition A: Raw ---
    print('\n[A] Raw prompt...')
    raw_ctx, raw_output, raw_time = execute_raw(question)
    raw_score = score_output(raw_ctx, raw_output)
    row['raw'] = {'output': raw_output[:500], 'score': raw_score, 'time': raw_time}
    print(f'    {format_score(raw_score)}  ({raw_time:.1f}s)')

    # --- Condition B: Single best template ---
    print('\n[B] Single template (root_cause_analyzer)...')
    try:
        single_ctx, single_output, single_time = execute_single_template(
            question, 'root_cause_analyzer'
        )
        single_score = score_output(single_ctx, single_output)
        row['single'] = {'output': single_output[:500], 'score': single_score, 'time': single_time}
        print(f'    {format_score(single_score)}  ({single_time:.1f}s)')
    except Exception as e:
        print(f'    ERROR: {e}')
        row['single'] = {'error': str(e)}

    # --- Condition C: Integrated template ---
    print('\n[C] Integrated template (suggest + integrate + execute)...')
    try:
        int_ctx, int_output, int_time, integration = execute_integrated(question)
        int_score = score_output(int_ctx, int_output)
        row['integrated'] = {
            'output': int_output[:500],
            'score': int_score,
            'time': int_time,
            'source_templates': integration.source_templates,
        }
        print(f'    Templates used: {integration.source_templates}')
        print(f'    {format_score(int_score)}  ({int_time:.1f}s)')
    except Exception as e:
        print(f'    ERROR: {e}')
        row['integrated'] = {'error': str(e)}

    results.append(row)

print(f'\n\nDone! {len(results)} questions tested.')

## Step 5 — Results Summary Table

In [ ]:
print(f'{"Question":<50} {"Raw":>8} {"Single":>8} {"Integrated":>11} {"Winner":>10}')
print('-' * 95)

raw_wins = single_wins = int_wins = 0

for row in results:
    q = row['question'][:48]
    r = row.get('raw', {}).get('score')
    s = row.get('single', {}).get('score')
    ig = row.get('integrated', {}).get('score')

    r_val = f'{r.overall:.1%}' if r else 'ERR'
    s_val = f'{s.overall:.1%}' if s else 'ERR'
    i_val = f'{ig.overall:.1%}' if ig else 'ERR'

    scores = []
    if r: scores.append(('Raw', r.overall))
    if s: scores.append(('Single', s.overall))
    if ig: scores.append(('Integrated', ig.overall))

    if scores:
        winner = max(scores, key=lambda x: x[1])[0]
        if winner == 'Raw': raw_wins += 1
        elif winner == 'Single': single_wins += 1
        else: int_wins += 1
    else:
        winner = '?'

    print(f'{q:<50} {r_val:>8} {s_val:>8} {i_val:>11} {winner:>10}')

print('-' * 95)
print(f'\nWins — Raw: {raw_wins} | Single Template: {single_wins} | Integrated: {int_wins}')

## Step 6 — Per-Dimension Breakdown

See which dimensions benefit most from integration.

In [ ]:
from collections import defaultdict

dim_totals = defaultdict(lambda: {'raw': [], 'single': [], 'integrated': []})

for row in results:
    for cond in ['raw', 'single', 'integrated']:
        score = row.get(cond, {}).get('score')
        if score:
            for dim, val in score.dimensions.items():
                dim_totals[dim.value][cond].append(val)

print(f'{"Dimension":<25} {"Raw":>8} {"Single":>8} {"Integrated":>11} {"Lift":>8}')
print('-' * 68)

for dim_name, conds in sorted(dim_totals.items()):
    r_avg = sum(conds['raw']) / len(conds['raw']) if conds['raw'] else 0
    s_avg = sum(conds['single']) / len(conds['single']) if conds['single'] else 0
    i_avg = sum(conds['integrated']) / len(conds['integrated']) if conds['integrated'] else 0
    lift = ((i_avg / max(r_avg, 0.01)) - 1) * 100
    label = dim_name.replace('_', ' ').title()
    print(f'{label:<25} {r_avg:>7.1%} {s_avg:>7.1%} {i_avg:>10.1%} {lift:>+7.1f}%')

## Step 7 — Deep Dive: Read the Actual Outputs

Pick a question index (0-based) to inspect the full outputs side by side.

In [ ]:
INSPECT_IDX = 0  # Change this to inspect different questions

row = results[INSPECT_IDX]
print(f'Question: {row["question"]}\n')

for cond, label in [('raw', 'A. RAW'), ('single', 'B. SINGLE TEMPLATE'), ('integrated', 'C. INTEGRATED')]:
    data = row.get(cond, {})
    if 'error' in data:
        print(f'\n--- {label} --- ERROR: {data["error"]}')
        continue
    score = data.get('score')
    print(f'\n{"="*70}')
    print(f'{label} — {format_score(score) if score else "no score"}')
    if cond == 'integrated' and 'source_templates' in data:
        print(f'Templates: {data["source_templates"]}')
    print('='*70)
    print(data.get('output', '(no output)')[:2000])
    print('...' if len(data.get('output', '')) > 2000 else '')

## Step 8 — CAI Measurement (Chain vs Single)

Use the built-in CAI engine to measure how much the chain amplifies quality over a single template.

In [ ]:
cai = ContextAmplificationIndex(provider=PROVIDER, eval_mode='heuristic')

# Pick one question to test CAI
cai_question = ACTIVE_QUESTIONS[0]
print(f'Question: {cai_question}\n')

# Single template CAI (template vs raw)
print('--- CAI: Template vs Raw ---')
cai_result = cai.measure(question=cai_question, template_name='root_cause_analyzer')
print(cai.report(cai_result))

In [ ]:
# Chain CAI (chain vs single best)
# First get the chain that was suggested for this question
if results and results[0].get('integrated', {}).get('source_templates'):
    chain = results[0]['integrated']['source_templates']
    print(f'Chain: {" → ".join(chain)}\n')
    print('--- CAI: Chain vs Single ---')
    chain_cai = cai.measure_chain(question=cai_question, chain=chain)
    print(cai.report(chain_cai))
else:
    print('No chain available. Run Step 4 first.')

## Step 9 — Context Quality Check

Score the context itself (not the output) to see if integrated templates produce higher quality prompts.

In [ ]:
print(f'{"Question":<50} {"Raw Ctx":>9} {"Single Ctx":>11} {"Integrated Ctx":>15}')
print('-' * 90)

for i, question in enumerate(ACTIVE_QUESTIONS):
    q_short = question[:48]

    # Raw context quality
    raw_ctx = Context(directive=Directive(content=question))
    raw_q = quality.evaluate(raw_ctx)

    # Single template context quality
    try:
        klass = get_pattern_class('root_cause_analyzer', include_enterprise=True)
        reg = PATTERN_BUILD_CONTEXT_REGISTRY.get('root_cause_analyzer', ('problem', {}))
        primary, defaults = reg
        params = dict(defaults)
        params[primary] = question
        single_ctx = klass().build_context(**params)
        single_q = quality.evaluate(single_ctx)
    except Exception:
        single_q = None

    # Integrated context quality
    int_score = results[i].get('integrated', {}).get('score')
    # Re-evaluate context quality from the stored integration
    int_q = None
    if results[i].get('integrated', {}).get('source_templates'):
        try:
            integration = integrator.suggest_and_integrate(question, provider=PROVIDER, max_patterns=4)
            int_ctx = integration.to_context()
            int_q = quality.evaluate(int_ctx)
        except Exception:
            pass

    r_val = f'{raw_q.overall:.1%}'
    s_val = f'{single_q.overall:.1%}' if single_q else 'ERR'
    i_val = f'{int_q.overall:.1%}' if int_q else 'ERR'
    print(f'{q_short:<50} {r_val:>9} {s_val:>11} {i_val:>15}')

print('\nNote: This scores the PROMPT quality, not the LLM output quality.')

---

## Step 10 — Save Results

Serialize results to JSON for the sprint report.

In [ ]:
from datetime import datetime


def serialize_results(results):
    out = []
    for row in results:
        r = {'question': row['question']}
        for cond in ['raw', 'single', 'integrated']:
            data = row.get(cond, {})
            if 'error' in data:
                r[cond] = {'error': data['error']}
            elif 'score' in data:
                s = data['score']
                r[cond] = {
                    'overall': round(s.overall, 4),
                    'dimensions': {d.value: round(v, 4) for d, v in s.dimensions.items()},
                    'time_seconds': round(data.get('time', 0), 2),
                }
                if 'source_templates' in data:
                    r[cond]['source_templates'] = data['source_templates']
        out.append(r)
    return out

report = {
    'sprint': 'Sprint 1 — Integration Effectiveness',
    'date': datetime.now().isoformat(),
    'provider': PROVIDER,
    'num_questions': len(results),
    'results': serialize_results(results),
}

outpath = 'sprint1_results.json'
with open(outpath, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Results saved to {outpath}')